<a href="https://colab.research.google.com/github/sileysa/Semester_5/blob/main/JS09/JS09_Tugas2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install KaggleHub

In [1]:
!pip install -q kagglehub annoy faiss-cpu hnswlib scikit-learn

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# Import library

In [2]:
import pandas as pd
import numpy as np
import time
import os
import glob
import kagglehub
import faiss
import hnswlib
import gc

from annoy import AnnoyIndex
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

print("Semua library berhasil di-import!")

Semua library berhasil di-import!


# Download dataset

In [3]:
path = kagglehub.dataset_download(
    "bwandowando/spotify-songs-with-attributes-and-lyrics"
)

print("Dataset berhasil di-download!")
print("Lokasi:", path)

100%|██████████| 894M/894M [00:52<00:00, 18.0MB/s]

Extracting files...


Dataset berhasil di-download!
Lokasi: /root/.cache/kagglehub/datasets/bwandowando/spotify-songs-with-attributes-and-lyrics/versions/20


# Periksa semua file CSV

In [4]:
csv_files = glob.glob(
    os.path.join(path, "**", "*.csv"),
    recursive=True
)

print("Jumlah file CSV:", len(csv_files))

for i, file in enumerate(csv_files):
    print(f"{i}. {os.path.basename(file)}")

Jumlah file CSV: 2
0. songs_with_lyrics_and_timestamps.csv
1. songs_with_attributes_and_lyrics.csv


# Cari CSV yang memiliki fitur musik

In [5]:
features = [
    'danceability',
    'energy',
    'loudness',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo'
]

file_terpilih = None

for file in csv_files:
    kolom = pd.read_csv(file, nrows=0).columns.tolist()

    print("\nFile:", os.path.basename(file))
    print("Kolom:", kolom)

    if all(fitur in kolom for fitur in features):
        file_terpilih = file

print("\nFile dengan fitur musik:", file_terpilih)


File: songs_with_lyrics_and_timestamps.csv
Kolom: ['id', 'startTimeMs', 'words']

File: songs_with_attributes_and_lyrics.csv
Kolom: ['id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics']

File dengan fitur musik: /root/.cache/kagglehub/datasets/bwandowando/spotify-songs-with-attributes-and-lyrics/versions/20/songs_with_attributes_and_lyrics.csv


# Load dataset pertama

In [6]:
if file_terpilih is None:
    print("File dengan sembilan fitur musik tidak ditemukan.")
else:
    df = pd.read_csv(file_terpilih)

    print("Dataset berhasil dimuat!")
    print("Nama file:", os.path.basename(file_terpilih))
    print("Ukuran dataset:", df.shape)

    display(df.head())

Dataset berhasil dimuat!
Nama file: songs_with_attributes_and_lyrics.csv
Ukuran dataset: (955320, 17)


,id,name,album_name,artists,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,lyrics
0,0Prct5TDjAnEgIqbxcldY9,!,UNDEN!ABLE,HELLYEAH,0.415,0.6050,7,-11.157,1,0.0575,0.00116,0.838000,0.4710,0.193,100.059,79500.0,"He said he came from Jamaica,\n he owned a cou..."
1,2ASl4wirkeYm3OWZxXKYuq,!!,NaN,Yxngxr1,0.788,0.6480,7,-9.135,0,0.3150,0.90000,0.000000,0.1760,0.287,79.998,114000.0,"Fucked a bitch, now she running with my kids\n..."
2,69lcggVPmOr9cvPx9kLiiN,!!! - Interlude,Where I Belong EP,Glowie,0.000,0.0354,7,-20.151,0,0.0000,0.90800,0.000000,0.4790,0.000,0.000,11413.0,"Oh, my God, I'm going crazy\n"
3,4U7dlZjg1s9pjdppqZy0fm,!!De Repente!!,Un Palo Al Agua (20 Grandes Canciones),Rosendo,0.657,0.8820,5,-6.340,1,0.0385,0.00740,0.000013,0.0474,0.939,123.588,198173.0,Continuamente se extraña la gente si no puede ...
4,4v1IBp3Y3rpkWmWzIlkYju,!!De Repente!!,Fuera De Lugar,Rosendo,0.659,0.8930,5,-8.531,1,0.0411,0.09220,0.000019,0.0534,0.951,123.600,199827.0,Continuamente se extraña la gente si no puede ...


# Periksa fitur dan missing values

In [7]:
print("Fitur yang digunakan:")
print(features)

print("\nMissing values:")
print(df[features].isnull().sum())

print("\nTipe data:")
print(df[features].dtypes)

Fitur yang digunakan:
['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

Missing values:
danceability        0
energy              0
loudness            0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
dtype: int64

Tipe data:
danceability        float64
energy              float64
loudness            float64
speechiness         float64
acousticness        float64
instrumentalness    float64
liveness            float64
valence             float64
tempo               float64
dtype: object


# Pilih fitur numerik dan normalisasi

In [8]:
# Mengambil fitur numerik
df_clean = df.dropna(subset=features).copy()

X = df_clean[features].values

# Standardisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype('float32')

print("Normalisasi berhasil!")
print("Jumlah data:", X_scaled.shape[0])
print("Jumlah fitur:", X_scaled.shape[1])
print("\nContoh hasil normalisasi:")
print(X_scaled[:5])

Normalisasi berhasil!
Jumlah data: 955320
Jumlah fitur: 9

Contoh hasil normalisasi:
[[-0.7993069  -0.19864811 -0.8763855  -0.28127217 -0.9037919   3.5533972
   1.2800521  -1.1735829  -0.75050366]
 [ 1.3975977  -0.01859905 -0.34315988  2.4896579   1.9789549  -0.38476917
  -0.22552979 -0.799778   -1.429702  ]
 [-3.2435842  -2.5836701  -3.248211   -0.9000235   2.0046124  -0.38476917
   1.3208814  -1.9410763  -4.138167  ]
 [ 0.6260307   0.9612028   0.3939151  -0.48572916 -0.8837791  -0.38470948
  -0.8818614   1.7929971   0.04610963]
 [ 0.6378103   1.0072619  -0.18387786 -0.45775083 -0.61180973 -0.3846794
  -0.8512394   1.840717    0.04651591]]


# Menentukan parameter eksperimen

In [9]:
# Parameter eksperimen
k = 10
jumlah_query = 100

# Memilih 100 track secara acak sebagai query
rng = np.random.default_rng(42)

query_indices = rng.choice(
    len(X_scaled),
    size=jumlah_query,
    replace=False
)

X_query = X_scaled[query_indices]

print("Jumlah data:", len(X_scaled))
print("Jumlah query:", len(X_query))
print("Jumlah nearest neighbors:", k)

Jumlah data: 955320
Jumlah query: 100
Jumlah nearest neighbors: 10


# Membangun Exact NN dan melakukan pencarian

In [10]:
# Membuat model Exact NN
exact_nn = NearestNeighbors(
    n_neighbors=k + 1,
    algorithm='brute',
    metric='euclidean',
    n_jobs=1
)

# Mengukur waktu pembangunan model
start_time = time.perf_counter()

exact_nn.fit(X_scaled)

exact_build_time = time.perf_counter() - start_time

# Mengukur waktu pencarian
start_time = time.perf_counter()

distances, indices = exact_nn.kneighbors(X_query)

exact_query_time = (
    time.perf_counter() - start_time
) / jumlah_query * 1000

# Menghilangkan track itu sendiri dari hasil pencarian
exact_results = []

for i in range(jumlah_query):
    hasil = [
        (idx, dist)
        for idx, dist in zip(indices[i], distances[i])
        if idx != query_indices[i]
    ]

    exact_results.append(hasil[:k])

print("Exact NN berhasil dijalankan!")
print("Build Time:", round(exact_build_time, 4), "detik")
print("Query Time:", round(exact_query_time, 4), "ms/query")

Exact NN berhasil dijalankan!
Build Time: 0.751 detik
Query Time: 20.0336 ms/query


# Menampilkan hasil pencarian

In [11]:
# Mengambil query pertama
query_id = query_indices[0]

print("Track yang dijadikan query:")
print(df_clean.iloc[query_id][['name', 'artists']])

# Mengambil hasil nearest neighbors
hasil_exact = exact_results[0]

hasil_df = df_clean.iloc[
    [idx for idx, dist in hasil_exact]
][['name', 'artists']].copy()

hasil_df['Distance'] = [
    dist for idx, dist in hasil_exact
]

hasil_df.index = range(1, k + 1)

print("\n10 Track Terdekat:")
display(hasil_df)

Track yang dijadikan query:
name       Kamli (From "Dhoom:3")
artists                    Pritam
Name: 417612, dtype: object

10 Track Terdekat:


,name,artists,Distance
1,"Kamli (From ""Dhoom:3"")",Sunidhi Chauhan,0.000000
2,How To Be Lonely - Bomba Estéreo Remix,"Rita Ora, Bomba Estéreo",0.442344
3,How To Be Lonely - Bomba Estéreo Remix,"Rita Ora, Bomba Estéreo",0.442344
4,Arriba las Banderas,Macaco,0.520409
5,"Vaa Saamy (From ""Annaatthe"")",D. Imman,0.545626
6,Ellas,Carlitos Junior;Sayian Jimmy,0.560780
7,Queuejumper,The Divine Comedy,0.589223
8,...A Livication,Iya Terra,0.595648
9,Dobla Rodilla,Don Omar,0.615396
10,En el aire,ChocQuibTown,0.621981


# Membuat fungsi pengujian ANNOY

In [12]:
def uji_annoy(X, X_query, query_indices, k=10):
    jumlah, dimensi = X.shape

    # Membuat indeks ANNOY
    index = AnnoyIndex(dimensi, 'euclidean')

    # Mengukur waktu pembangunan indeks
    start_time = time.perf_counter()

    for i, vector in enumerate(X):
        index.add_item(i, vector)

    index.build(10)

    build_time = time.perf_counter() - start_time

    # Mengukur waktu pencarian
    hasil = []

    start_time = time.perf_counter()

    for query in X_query:
        indices = index.get_nns_by_vector(
            query.tolist(),
            k + 1
        )

        hasil.append(indices)

    query_time = (
        time.perf_counter() - start_time
    ) / len(X_query) * 1000

    # Menghilangkan track query sendiri
    hasil_bersih = []

    for i, neighbors in enumerate(hasil):
        neighbors = [
            idx for idx in neighbors
            if idx != query_indices[i]
        ]

        hasil_bersih.append(neighbors[:k])

    # Menyimpan indeks
    index.save('spotify_annoy.ann')

    index_size = (
        os.path.getsize('spotify_annoy.ann') / (1024 ** 2)
    )

    return {
        'Build Time (s)': build_time,
        'Query Time (ms)': query_time,
        'Index Size (MB)': index_size,
        'Results': hasil_bersih
    }

# Menjalankan ANNOY

In [13]:
hasil_annoy = uji_annoy(
    X_scaled,
    X_query,
    query_indices,
    k
)

print("Hasil Pengujian ANNOY:")
print("Build Time:", round(hasil_annoy['Build Time (s)'], 4), "detik")
print("Query Time:", round(hasil_annoy['Query Time (ms)'], 4), "ms/query")
print("Index Size:", round(hasil_annoy['Index Size (MB)'], 4), "MB")

Hasil Pengujian ANNOY:
Build Time: 23.971 detik
Query Time: 0.8224 ms/query
Index Size: 169.1077 MB


# Menampilkan 10 hasil pencarian ANNOY

In [14]:
hasil_annoy_df = df_clean.iloc[
    hasil_annoy['Results'][0]
][['name', 'artists']].copy()

hasil_annoy_df['Distance'] = [
    np.linalg.norm(
        X_scaled[idx] - X_query[0]
    )
    for idx in hasil_annoy['Results'][0]
]

hasil_annoy_df.index = range(1, len(hasil_annoy_df) + 1)

print("10 Track Terdekat menggunakan ANNOY:")
display(hasil_annoy_df)

10 Track Terdekat menggunakan ANNOY:


,name,artists,Distance
1,"Kamli (From ""Dhoom:3"")",Sunidhi Chauhan,0.000000
2,How To Be Lonely - Bomba Estéreo Remix,"Rita Ora, Bomba Estéreo",0.442344
3,How To Be Lonely - Bomba Estéreo Remix,"Rita Ora, Bomba Estéreo",0.442344
4,Arriba las Banderas,Macaco,0.520409
5,Queuejumper,The Divine Comedy,0.589223
6,En el aire,ChocQuibTown,0.621981
7,Boom Bye Bye,"Héctor ""El Father""",0.638273
8,Hello Hello Hello - Polo & Pan Remix,Remi Wolf,0.650526
9,"迷宮 - (電視劇 ""拆局專家"" 主題曲)",Alfred Hui,0.655966
10,Sovereign,"The New Life Community Choir, John P. Kee",0.681397


# Menghitung Recall@10

In [15]:
# Mengambil hasil Exact NN tanpa track query sendiri
exact_results_clean = []

for i in range(jumlah_query):
    neighbors = [
        idx for idx in indices[i]
        if idx != query_indices[i]
    ]

    exact_results_clean.append(neighbors[:k])


# Menghitung Recall@10 ANNOY
recall_annoy = []

for i in range(jumlah_query):
    exact_set = set(exact_results_clean[i])
    annoy_set = set(hasil_annoy['Results'][i])

    recall = len(exact_set.intersection(annoy_set)) / k

    recall_annoy.append(recall)


mean_recall_annoy = np.mean(recall_annoy)

print("Recall@10 ANNOY:", round(mean_recall_annoy, 4))

Recall@10 ANNOY: 0.849


# Membuat fungsi pengujian FAISS

In [16]:
def uji_faiss(X, X_query, query_indices, k=10):
    jumlah, dimensi = X.shape

    # Membuat index FAISS
    nlist = 1000

    quantizer = faiss.IndexFlatL2(dimensi)

    index = faiss.IndexIVFFlat(
        quantizer,
        dimensi,
        nlist,
        faiss.METRIC_L2
    )

    # Mengatur jumlah cluster yang diperiksa
    index.nprobe = 50

    # Mengukur waktu pembangunan indeks
    start_time = time.perf_counter()

    index.train(X)
    index.add(X)

    build_time = time.perf_counter() - start_time

    # Mengukur waktu pencarian
    start_time = time.perf_counter()

    distances, indices = index.search(
        X_query,
        k + 1
    )

    query_time = (
        time.perf_counter() - start_time
    ) / len(X_query) * 1000

    # Menghilangkan track query sendiri
    hasil_bersih = []

    for i in range(len(X_query)):
        neighbors = [
            idx for idx in indices[i]
            if idx != query_indices[i]
        ]

        hasil_bersih.append(neighbors[:k])

    # Menyimpan indeks FAISS
    faiss.write_index(index, 'spotify_faiss.index')

    index_size = (
        os.path.getsize('spotify_faiss.index') / (1024 ** 2)
    )

    return {
        'Build Time (s)': build_time,
        'Query Time (ms)': query_time,
        'Index Size (MB)': index_size,
        'Results': hasil_bersih
    }

# Menjalankan FAISS

In [17]:
hasil_faiss = uji_faiss(
    X_scaled,
    X_query,
    query_indices,
    k
)

print("Hasil Pengujian FAISS:")
print("Build Time:", round(hasil_faiss['Build Time (s)'], 4), "detik")
print("Query Time:", round(hasil_faiss['Query Time (ms)'], 4), "ms/query")
print("Index Size:", round(hasil_faiss['Index Size (MB)'], 4), "MB")

Hasil Pengujian FAISS:
Build Time: 3.2693 detik
Query Time: 1.0275 ms/query
Index Size: 40.1289 MB


# Menampilkan 10 hasil pencarian FAISS

In [18]:
hasil_faiss_df = df_clean.iloc[
    hasil_faiss['Results'][0]
][['name', 'artists']].copy()

hasil_faiss_df['Distance'] = [
    np.linalg.norm(
        X_scaled[idx] - X_query[0]
    )
    for idx in hasil_faiss['Results'][0]
]

hasil_faiss_df.index = range(1, len(hasil_faiss_df) + 1)

print("10 Track Terdekat menggunakan FAISS:")
display(hasil_faiss_df)

10 Track Terdekat menggunakan FAISS:


,name,artists,Distance
1,"Kamli (From ""Dhoom:3"")",Sunidhi Chauhan,0.000000
2,How To Be Lonely - Bomba Estéreo Remix,"Rita Ora, Bomba Estéreo",0.442344
3,How To Be Lonely - Bomba Estéreo Remix,"Rita Ora, Bomba Estéreo",0.442344
4,Arriba las Banderas,Macaco,0.520409
5,"Vaa Saamy (From ""Annaatthe"")",D. Imman,0.545626
6,Ellas,Carlitos Junior;Sayian Jimmy,0.560780
7,Queuejumper,The Divine Comedy,0.589223
8,...A Livication,Iya Terra,0.595648
9,Dobla Rodilla,Don Omar,0.615396
10,En el aire,ChocQuibTown,0.621981


# Menghitung Recall@10 FAISS

In [19]:
recall_faiss = []

for i in range(jumlah_query):
    exact_set = set(exact_results_clean[i])
    faiss_set = set(hasil_faiss['Results'][i])

    recall = len(exact_set.intersection(faiss_set)) / k

    recall_faiss.append(recall)

mean_recall_faiss = np.mean(recall_faiss)

print("Recall@10 FAISS:", round(mean_recall_faiss, 4))

Recall@10 FAISS: 0.998


# Membuat fungsi pengujian HNSWLIB

In [20]:
def uji_hnsw(X, X_query, query_indices, k=10):
    jumlah, dimensi = X.shape

    # Membuat indeks HNSW
    index = hnswlib.Index(
        space='l2',
        dim=dimensi
    )

    # Inisialisasi indeks
    index.init_index(
        max_elements=jumlah,
        ef_construction=200,
        M=16,
        random_seed=42
    )

    index.set_num_threads(4)

    # Mengukur waktu pembangunan indeks
    start_time = time.perf_counter()

    index.add_items(
        X,
        np.arange(jumlah)
    )

    build_time = time.perf_counter() - start_time

    # Mengatur parameter pencarian
    index.set_ef(200)

    # Mengukur waktu pencarian
    start_time = time.perf_counter()

    indices, distances = index.knn_query(
        X_query,
        k=k + 1,
        num_threads=1
    )

    query_time = (
        time.perf_counter() - start_time
    ) / len(X_query) * 1000

    # Menghilangkan track query sendiri
    hasil_bersih = []

    for i in range(len(X_query)):
        neighbors = [
            idx for idx in indices[i]
            if idx != query_indices[i]
        ]

        hasil_bersih.append(neighbors[:k])

    # Menyimpan indeks
    index.save_index('spotify_hnsw.bin')

    index_size = (
        os.path.getsize('spotify_hnsw.bin') / (1024 ** 2)
    )

    return {
        'Build Time (s)': build_time,
        'Query Time (ms)': query_time,
        'Index Size (MB)': index_size,
        'Results': hasil_bersih
    }

# Menjalankan HNSWLIB

In [21]:
hasil_hnsw = uji_hnsw(
    X_scaled,
    X_query,
    query_indices,
    k
)

print("Hasil Pengujian HNSWLIB:")
print("Build Time:", round(hasil_hnsw['Build Time (s)'], 4), "detik")
print("Query Time:", round(hasil_hnsw['Query Time (ms)'], 4), "ms/query")
print("Index Size:", round(hasil_hnsw['Index Size (MB)'], 4), "MB")

Hasil Pengujian HNSWLIB:
Build Time: 237.9522 detik
Query Time: 0.2658 ms/query
Index Size: 168.1125 MB


# Menampilkan 10 hasil pencarian HNSWLIB

In [22]:
hasil_hnsw_df = df_clean.iloc[
    hasil_hnsw['Results'][0]
][['name', 'artists']].copy()

hasil_hnsw_df['Distance'] = [
    np.linalg.norm(
        X_scaled[idx] - X_query[0]
    )
    for idx in hasil_hnsw['Results'][0]
]

hasil_hnsw_df.index = range(1, len(hasil_hnsw_df) + 1)

print("10 Track Terdekat menggunakan HNSWLIB:")
display(hasil_hnsw_df)

10 Track Terdekat menggunakan HNSWLIB:


,name,artists,Distance
1,"Kamli (From ""Dhoom:3"")",Sunidhi Chauhan,0.000000
2,How To Be Lonely - Bomba Estéreo Remix,"Rita Ora, Bomba Estéreo",0.442344
3,How To Be Lonely - Bomba Estéreo Remix,"Rita Ora, Bomba Estéreo",0.442344
4,Arriba las Banderas,Macaco,0.520409
5,"Vaa Saamy (From ""Annaatthe"")",D. Imman,0.545626
6,Ellas,Carlitos Junior;Sayian Jimmy,0.560780
7,Queuejumper,The Divine Comedy,0.589223
8,...A Livication,Iya Terra,0.595648
9,Dobla Rodilla,Don Omar,0.615396
10,En el aire,ChocQuibTown,0.621981


# Menghitung Recall@10 HNSWLIB

In [23]:
recall_hnsw = []

for i in range(jumlah_query):
    exact_set = set(exact_results_clean[i])
    hnsw_set = set(hasil_hnsw['Results'][i])

    recall = len(exact_set.intersection(hnsw_set)) / k

    recall_hnsw.append(recall)

mean_recall_hnsw = np.mean(recall_hnsw)

print("Recall@10 HNSWLIB:", round(mean_recall_hnsw, 4))

Recall@10 HNSWLIB: 0.997


### Kesimpulan

Berdasarkan percobaan penggunaan Exact Nearest Neighbor (Exact NN), ANNOY, FAISS, dan HNSWLIB pada dataset Micro Spotify yang terdiri dari 955.320 data lagu dan 9 fitur numerik, dapat disimpulkan bahwa setiap algoritma memiliki karakteristik dan keunggulan masing-masing.

Exact NN digunakan sebagai metode pencarian acuan karena melakukan pencarian secara langsung terhadap seluruh data. Metode ini menghasilkan Recall@10 sebesar 1,000, tetapi membutuhkan waktu pencarian 20,0336 ms/query.

ANNOY memiliki waktu pencarian sebesar 0,8224 ms/query dengan Recall@10 sebesar 0,849. FAISS menghasilkan performa pencarian yang mendekati Exact NN dengan Recall@10 sebesar 0,998 dan ukuran indeks paling kecil, yaitu 40,1289 MB. Sementara itu, HNSWLIB memiliki waktu pencarian paling cepat, yaitu 0,2658 ms/query, dengan Recall@10 sebesar 0,997, tetapi membutuhkan waktu pembangunan indeks paling lama, yaitu 237,9522 detik.

Dengan demikian, FAISS menjadi pilihan yang baik apabila mempertimbangkan keseimbangan antara akurasi pencarian dan penggunaan penyimpanan. HNSWLIB lebih sesuai apabila kecepatan pencarian menjadi prioritas utama, sedangkan ANNOY menawarkan alternatif pencarian yang lebih cepat dibandingkan Exact NN dengan proses pembangunan indeks yang relatif lebih singkat dibandingkan HNSWLIB.